In [0]:
checkpointLocation ="/Volumes/iot-telemetry/raw/iot-telemetry-files/configs/telemetry/checkpoint/"
schemaLocation = "/Volumes/iot-telemetry/raw/iot-telemetry-files/configs/telemetry/schema/"
filePath = "/Volumes/iot-telemetry/raw/iot-telemetry-files/telemetry/"

In [0]:
%run "./common_methods"

In [0]:
table_name="`iot-telemetry`.raw.telemetry"

In [0]:
from pyspark.sql.functions import current_timestamp

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schemaLocation)
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        # .schema(device_schema)
        .load(filePath)
)

df = df.withColumn(
    "created_at",
    current_timestamp()
)

try:
    query = (
        df.writeStream
            .option("checkpointLocation", checkpointLocation)
            .trigger(availableNow=True)
            .toTable(table_name)
    )

except Exception as e:
    print(e)